In [1]:
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE_DIR = "/content/drive/MyDrive/running_coach"
sys.path.insert(0, f"{BASE_DIR}/tools")
from calculate_pace_zones import classify_workouts, format_pace
from parse_workout_data import load_workouts

workouts = load_workouts(f"{BASE_DIR}/data/processed/workouts_normalized.json", days=90)
enriched = classify_workouts(workouts)

structured  = [w for w in enriched if w["classification"]["is_structured"]]
needs_input = [w for w in enriched if w["classification"]["needs_input"]]
print(f"Structured: {len(structured)} | Needs input: {len(needs_input)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Structured: 24 | Needs input: 0


In [2]:
from parse_workout_data import load_workouts

workouts = load_workouts(f"{BASE_DIR}/data/processed/workouts_normalized.json", days=90)

# Print all activity names so we can see the actual format
for w in sorted(workouts, key=lambda x: x['date'], reverse=True)[:20]:
    print(f"{w['date']} | {w.get('garmin_name', w.get('name', ''))}")

2026-04-26 | Halifax Running
2026-04-25 | Halifax - 2026-04-25
2026-04-24 | Halifax Running
2026-04-23 | Halifax Running
2026-04-22 | Halifax - 2026-04-22
2026-04-21 | Halifax - Easy w/ strides
2026-04-19 | Halifax Running
2026-04-18 | Halifax - 2026-04-17
2026-04-18 | Halifax - 2026-04-17
2026-04-16 | Halifax Running
2026-04-16 | Halifax Running
2026-04-15 | Halifax - 2026-04-15
2026-04-14 | Halifax Running
2026-04-12 | Halifax Running
2026-04-11 | Morning Run
2026-04-10 | Halifax - 2026-04-10
2026-04-08 | Halifax Running
2026-04-07 | Halifax Running
2026-04-05 | Halifax - Easy w/ strides
2026-04-05 | Halifax - Easy w/ strides


In [3]:
needs_input = [w for w in enriched if w["classification"]["needs_input"]]
for w in needs_input:
    print(f"{w['date']} | {w.get('garmin_name', '')} | {w.get('name', '')}")

In [4]:
import sys
sys.path.insert(0, f"{BASE_DIR}/tools")
from parse_workout_data import load_workouts
from calculate_training_load import (
    get_current_metrics, get_recent_trend,
    check_recovery_alert, weekly_load_summary,
    check_mileage_rule, format_metrics_report
)

workouts = load_workouts(
    f"{BASE_DIR}/data/processed/workouts_normalized.json",
    days=180
)

metrics = get_current_metrics(workouts)
trend   = get_recent_trend(workouts, days=14)
alert   = check_recovery_alert(workouts)
mileage = check_mileage_rule(workouts)

print(format_metrics_report(metrics, trend, alert, mileage))

  ℹ️  1 record(s) skipped (no training_load — not Garmin-enriched). Treated as 0 load.
=== Training Load Report — 2026-04-27 ===
  ATL (fatigue):   90.1
  CTL (fitness):   96.1
  TSB (form):      6.1  →  Fresh — good form
  Fitness level:   High fitness

  14-day trend:
    ATL:  123.9 → 90.1  (falling)
    CTL:  97.7 → 96.1  (falling)
    TSB:  -26.2 → 6.1  (rising)
    Total load:     1293
    Avg daily load: 92.3
    Peak load:      320.0
    Rest days:      3 / 14

  Mileage rule: ℹ️  Week just started (day 1/7) — no sessions logged yet. Last week total load: 607. 10% rule target for this week: ≤ 668.


In [5]:
import sqlite3, sys
sys.path.insert(0, f"{BASE_DIR}/tools")
from query_garmin_db import (
    get_recent_snapshots, get_race_predictions,
    format_recovery_context, format_race_predictions
)

conn = sqlite3.connect(f"{BASE_DIR}/data/raw/garmin/garmin.db")

snapshots = get_recent_snapshots(conn, days=3)
print(format_recovery_context(snapshots))

pred = get_race_predictions(conn)
print(format_race_predictions(pred))

conn.close()

=== Recovery Context ===

--- 2026-04-27 ---

--- 2026-04-26 ---
  Recovery score (composite): 65/100
  Training readiness: 54.0 — Low ⚠️
    Feedback: RECOVERY_IN_PROGRESS
    Recovery time remaining: 1794h
    HRV factor: GOOD
    Sleep history: MODERATE
  HRV: None (weekly avg: 92.0) — Balanced ✅, outside baseline ⚠️
  Sleep: 8.48h total (deep: 1.33h, REM: 1.73h) — Good duration
    Feedback: POSITIVE_LONG_AND_CONTINUOUS
  Body battery at wake: None — No data (peak: None, low: None)
  Stress: avg None, max 96 — Unknown
  Resting HR: 44 bpm

--- 2026-04-25 ---
  Recovery score (composite): 55/100
  Training readiness: 67.0 — Moderate
    Feedback: LISTEN_TO_YOUR_BODY
    Recovery time remaining: 1h
    HRV factor: MODERATE
    Sleep history: GOOD
  HRV: None (weekly avg: 90.0) — Unbalanced ⚠️, outside baseline ⚠️
  Sleep: 6.03h total (deep: 1.27h, REM: 0.73h) — Short sleep ⚠️
    Feedback: POSITIVE_DEEP
  Body battery at wake: None — No data (peak: None, low: None)
  Stress: avg None